In [1]:
from pathlib import Path
import json
import math
import shutil
import sys
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

code_file=Path("/kaggle/input/datasets/risha8750/phase2-core3/phase2_core.py")
assert code_file.exists()
shutil.copy2(code_file,"/kaggle/working/phase2_core.py")
sys.path.insert(0,"/kaggle/working")
from phase2_core import *
seed_everything(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type=="cuda"
torch.set_float32_matmul_precision("high")
{"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__}

{'gpu': 'Tesla T4', 'torch': '2.10.0+cu128'}

In [2]:
cache_root=next(p.parent for p in Path("/kaggle/input").rglob("index.csv") if p.parent.name=="phase2_cache")
out_root=Path("/kaggle/working/phase2_outputs")
out_root.mkdir(parents=True,exist_ok=True)
corner_dir=out_root/"corner"
edge_dir=out_root/"edge"
mesh_dir=out_root/"meshes"
for path in [corner_dir,edge_dir,mesh_dir]:
    path.mkdir(parents=True,exist_ok=True)
train_set=RoofDataset(cache_root,"train",augment=False)
val_set=RoofDataset(cache_root,"val")
test_set=RoofDataset(cache_root,"test")
len(train_set),len(val_set),len(test_set)

(25579, 3170, 3235)

In [3]:
batch_size=16
workers=2
train_loader=DataLoader(train_set,batch_size=batch_size,shuffle=True,num_workers=workers,pin_memory=True,drop_last=True,persistent_workers=True)
val_loader=DataLoader(val_set,batch_size=batch_size,shuffle=False,num_workers=workers,pin_memory=True,persistent_workers=True)
batch=next(iter(train_loader))
assert batch["points"].shape[1:]==(2048,8)
assert torch.isfinite(batch["points"]).all()
{k:tuple(v.shape) for k,v in batch.items() if torch.is_tensor(v)}

{'points': (16, 2048, 8),
 'corners': (16, 64, 3),
 'corner_count': (16,),
 'edges': (16, 128, 2),
 'edge_count': (16,),
 'center': (16, 3),
 'scale': (16,)}

In [4]:
def augment_batch(points,corners,counts):
    batch_size=points.shape[0]
    angles=(torch.rand(batch_size,device=points.device)*2-1)*math.pi
    cos=torch.cos(angles)
    sin=torch.sin(angles)
    zeros=torch.zeros_like(cos)
    ones=torch.ones_like(cos)
    rot=torch.stack([cos,-sin,zeros,sin,cos,zeros,zeros,zeros,ones],dim=1).view(batch_size,3,3)
    points[:,:,:3]=torch.bmm(points[:,:,:3],rot.transpose(1,2))
    for i in range(batch_size):
        n=int(counts[i])
        corners[i,:n]=corners[i,:n]@rot[i].T
    flip=(torch.rand(batch_size,device=points.device)<0.5).float()*-2+1
    points[:,:,0]*=flip[:,None]
    for i in range(batch_size):
        n=int(counts[i])
        corners[i,:n,0]*=flip[i]
    points[:,:,:3]+=torch.randn_like(points[:,:,:3])*0.003
    return points,corners

In [5]:
corner_model=CornerPointNet2().to(device)
trial=batch["points"][:2].to(device)
with torch.no_grad():
    trial_logits,trial_offsets,trial_features=corner_model(trial)
assert trial_logits.shape==(2,2048)
assert trial_offsets.shape==(2,2048,3)
assert trial_features.shape==(2,2048,128)
assert torch.isfinite(trial_logits).all()
{"parameters":sum(x.numel() for x in corner_model.parameters()),"logits":tuple(trial_logits.shape),"offsets":tuple(trial_offsets.shape),"features":tuple(trial_features.shape)}

{'parameters': 807940,
 'logits': (2, 2048),
 'offsets': (2, 2048, 3),
 'features': (2, 2048, 128)}

In [6]:
for path in [corner_dir/"corner_last.pt",corner_dir/"corner_best.pt"]:
    if path.exists():
        path.unlink()

seed_everything(42)
corner_model=CornerPointNet2().to(device)
corner_target_epochs=40
corner_epochs_this_run=5
corner_radius_m=0.35
corner_optimizer=torch.optim.AdamW(corner_model.parameters(),lr=0.0001,weight_decay=0.0001)
corner_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(corner_optimizer,T_max=corner_target_epochs,eta_min=0.00001)
corner_start=0
corner_history=[]

In [7]:
def corner_epoch(loader,training):
    corner_model.train(training)
    totals=np.zeros(4,dtype=np.float64)
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in loader:
            points=batch["points"].to(device,non_blocking=True)
            corners=batch["corners"].to(device,non_blocking=True)
            counts=batch["corner_count"].to(device,non_blocking=True)
            scales=batch["scale"].to(device,non_blocking=True)
            labels,offset_targets=corner_targets(points,corners,counts,scales,corner_radius_m)
            if training:
                corner_optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits,pred_offsets,_=corner_model(points)
                loss,cls_loss,reg_loss=corner_loss(logits,offset_targets,pred_offsets,labels)
            if training:
                corner_scaler.scale(loss).backward()
                corner_scaler.unscale_(corner_optimizer)
                torch.nn.utils.clip_grad_norm_(corner_model.parameters(),5.0)
                corner_scaler.step(corner_optimizer)
                corner_scaler.update()
            pred=(logits.detach()>0).float()
            tp=(pred*labels).sum().item()
            totals+=np.array([loss.item()*len(points),cls_loss.item()*len(points),reg_loss.item()*len(points),len(points)])
    return {"loss":totals[0]/totals[3],"cls":totals[1]/totals[3],"offset":totals[2]/totals[3]}

In [8]:
diagnostic_batch=next(iter(train_loader))
diagnostic_points=diagnostic_batch["points"].to(device).float()
diagnostic_corners=diagnostic_batch["corners"].to(device).float()
diagnostic_counts=diagnostic_batch["corner_count"].to(device)
diagnostic_scales=diagnostic_batch["scale"].to(device).float()

print("scale_min",diagnostic_scales.min().item())
print("scale_max",diagnostic_scales.max().item())
print("scale_has_nan",torch.isnan(diagnostic_scales).any().item())
print("points_has_nan",torch.isnan(diagnostic_points).any().item())
print("corners_has_nan",torch.isnan(diagnostic_corners).any().item())

diagnostic_labels,diagnostic_offsets=corner_targets(diagnostic_points,diagnostic_corners,diagnostic_counts,diagnostic_scales,corner_radius_m)
print("labels_has_nan",torch.isnan(diagnostic_labels).any().item())
print("offsets_has_nan",torch.isnan(diagnostic_offsets).any().item())
print("offsets_max_abs",diagnostic_offsets.abs().max().item())

diagnostic_logits,diagnostic_pred_offsets,_=corner_model(diagnostic_points)
print("logits_has_nan",torch.isnan(diagnostic_logits).any().item())
print("logits_max_abs",diagnostic_logits.abs().max().item())

scale_min 5.0354905128479
scale_max 19.592344284057617
scale_has_nan False
points_has_nan False
corners_has_nan False
labels_has_nan False
offsets_has_nan False
offsets_max_abs 36.50166320800781
logits_has_nan False
logits_max_abs 1.8819626569747925


In [9]:
def corner_epoch(loader,training):
    corner_model.train(training)
    total_loss=0.0
    total_cls=0.0
    total_offset=0.0
    total_samples=0

    context=torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            points=batch["points"].to(device,non_blocking=True).float()
            corners=batch["corners"].to(device,non_blocking=True).float()
            counts=batch["corner_count"].to(device,non_blocking=True)
            scales=batch["scale"].to(device,non_blocking=True).float()

            if training:
                points,corners=augment_batch(points,corners,counts)

            labels,offset_targets=corner_targets(points,corners,counts,scales,corner_radius_m)

            if training:
                corner_optimizer.zero_grad(set_to_none=True)

            logits,pred_offsets,_=corner_model(points)
            loss,cls_loss,reg_loss=corner_loss(logits,offset_targets,pred_offsets,labels)

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite corner loss encountered")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(corner_model.parameters(),1.0)
                corner_optimizer.step()

            size=len(points)
            total_loss+=loss.item()*size
            total_cls+=cls_loss.item()*size
            total_offset+=reg_loss.item()*size
            total_samples+=size

    return {
        "loss":total_loss/max(total_samples,1),
        "cls":total_cls/max(total_samples,1),
        "offset":total_offset/max(total_samples,1)
    }

In [10]:
corner_stop=min(corner_start+corner_epochs_this_run,corner_target_epochs)
best_val=float("inf")

for epoch in range(corner_start,corner_stop):
    train_metrics=corner_epoch(train_loader,True)
    val_metrics=corner_epoch(val_loader,False)
    corner_scheduler.step()

    row={
        "epoch":epoch,
        "train_loss":train_metrics["loss"],
        "train_cls":train_metrics["cls"],
        "train_offset":train_metrics["offset"],
        "val_loss":val_metrics["loss"],
        "val_cls":val_metrics["cls"],
        "val_offset":val_metrics["offset"],
        "lr":corner_optimizer.param_groups[0]["lr"]
    }
    corner_history.append(row)

    state={
        "epoch":epoch,
        "model":corner_model.state_dict(),
        "optimizer":corner_optimizer.state_dict(),
        "scheduler":corner_scheduler.state_dict(),
        "history":corner_history
    }

    torch.save(state,corner_dir/"corner_last.pt")

    if val_metrics["loss"]<best_val:
        best_val=val_metrics["loss"]
        torch.save(state,corner_dir/"corner_best.pt")

pd.DataFrame(corner_history).tail()

,epoch,train_loss,train_cls,train_offset,val_loss,val_cls,val_offset,lr
0,0,0.096322,0.015950,0.080372,0.084671,0.012523,0.072148,0.000100
1,1,0.082868,0.012020,0.070848,0.080211,0.011496,0.068715,0.000099
2,2,0.079350,0.011307,0.068043,0.077524,0.011054,0.066471,0.000099
3,3,0.076686,0.010830,0.065856,0.074963,0.010536,0.064428,0.000098
4,4,0.074256,0.010277,0.063979,0.072389,0.009904,0.062485,0.000097


In [11]:
best_corner=torch.load(corner_dir/"corner_best.pt",map_location=device,weights_only=False)
corner_model.load_state_dict(best_corner["model"])
corner_model.eval()
geometry_rows=[]
with torch.no_grad():
    for sample_no,sample in enumerate(val_set):
        if sample_no>=500:
            break
        points=sample["points"].to(device)
        logits,offsets,_=corner_model(points[None])
        pred,_=predict_corners(points,logits[0],offsets[0],sample["scale"],0.5,corner_radius_m,0.25,2,64)
        scale=float(sample["scale"])
        truth=sample["corners"][:sample["corner_count"]].numpy()*scale
        pred=pred*scale
        mapping,tp,total=match_corners(pred,truth,0.5)
        geometry_rows.append({"corner_tp":tp,"corner_pred":len(pred),"corner_true":len(truth),"corner_offset_sum":total,"edge_tp":0,"edge_pred":0,"edge_true":0})
corner_geometry=finalize_counts(geometry_rows)
assert np.isfinite(list(corner_geometry.values())).all()
corner_geometry

{'ACO': 0.19405442290332006,
 'CP': 0.5823767724510466,
 'CR': 0.2928692699490662,
 'C_F1': 0.38974243108901946,
 'EP': 0.0,
 'ER': 0.0,
 'E_F1': 0.0}

In [12]:
for parameter in corner_model.parameters():
    parameter.requires_grad=False

seed_everything(42)
edge_model=EdgeGraphNet().to(device)
edge_lr=0.0001
edge_weight_decay=0.0001
edge_optimizer=torch.optim.AdamW(edge_model.parameters(),lr=edge_lr,weight_decay=edge_weight_decay)
edge_target_epochs=30
edge_epochs_this_run=5
edge_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(edge_optimizer,T_max=edge_target_epochs,eta_min=0.00001)
edge_start=0
edge_history=[]

In [13]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu,memory.used,memory.total","--format=csv"],capture_output=True,text=True).stdout)

utilization.gpu [%], memory.used [MiB], memory.total [MiB]
38 %, 3637 MiB, 15360 MiB
0 %, 3 MiB, 15360 MiB



In [14]:
def edge_epoch(loader,training):
    edge_model.train(training)
    total_loss=0.0
    total_graphs=0
    context=torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            points=batch["points"].to(device,non_blocking=True).float()
            corners=batch["corners"].to(device,non_blocking=True).float()
            counts=batch["corner_count"].to(device,non_blocking=True)
            edges=batch["edges"].to(device,non_blocking=True)
            edge_counts=batch["edge_count"].to(device,non_blocking=True)

            with torch.no_grad():
                _,_,features=corner_model(points)

            if training:
                edge_optimizer.zero_grad(set_to_none=True)

            losses=[]
            for i in range(len(points)):
                n=int(counts[i])
                if n<2:
                    continue
                xyz=corners[i,:n]
                score=torch.ones(n,device=device)
                node_features=interpolate_features(points[i],features[i],xyz.detach().cpu().numpy())
                pairs,logits=edge_model(xyz,score,node_features)
                labels=edge_labels(pairs,edges[i],int(edge_counts[i]))
                losses.append(focal_binary(logits,labels,0.75,2.0))

            if not losses:
                continue

            loss=torch.stack(losses).mean()

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite edge loss")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(edge_model.parameters(),1.0)
                edge_optimizer.step()

            total_loss+=loss.item()*len(points)
            total_graphs+=len(points)

    return total_loss/max(total_graphs,1)

In [15]:
edge_stop=min(edge_start+edge_epochs_this_run,edge_target_epochs)
best_val=min((x["val_loss"] for x in edge_history if np.isfinite(x["val_loss"])),default=float("inf"))

for epoch in range(edge_start,edge_stop):
    train_loss=edge_epoch(train_loader,True)
    val_loss=edge_epoch(val_loader,False)

    if not np.isfinite(train_loss):
        raise RuntimeError(f"Non-finite edge training loss at epoch {epoch}")
    if not np.isfinite(val_loss):
        raise RuntimeError(f"Non-finite edge validation loss at epoch {epoch}")

    edge_scheduler.step()
    row={"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"lr":edge_optimizer.param_groups[0]["lr"]}
    edge_history.append(row)

    state={"epoch":epoch,"model":edge_model.state_dict(),"optimizer":edge_optimizer.state_dict(),"scheduler":edge_scheduler.state_dict(),"history":edge_history}
    torch.save(state,edge_dir/"edge_last.pt")

    if val_loss<best_val:
        best_val=val_loss
        torch.save(state,edge_dir/"edge_best.pt")

assert (edge_dir/"edge_last.pt").exists()
assert (edge_dir/"edge_best.pt").exists()
pd.DataFrame(edge_history).tail()

,epoch,train_loss,val_loss,lr
0,0,0.041519,0.033337,0.000100
1,1,0.030726,0.028736,0.000099
2,2,0.026811,0.025170,0.000098
3,3,0.024264,0.023425,0.000096
4,4,0.022577,0.022270,0.000094


In [16]:
edge_resume=sorted(Path("/kaggle/input").rglob("edge_last.pt"))
if edge_resume:
    state=torch.load(edge_resume[-1],map_location=device,weights_only=False)
    edge_model.load_state_dict(state["model"])
    edge_optimizer.load_state_dict(state["optimizer"])
    edge_scheduler.load_state_dict(state["scheduler"])
    edge_start=int(state["epoch"])+1
    edge_history=state["history"]
edge_start

0

In [17]:
best_edge=torch.load(edge_dir/"edge_best.pt",map_location=device,weights_only=False)
edge_model.load_state_dict(best_edge["model"])
corner_model.eval()
edge_model.eval()

def reconstruct(sample,edge_threshold=0.5):
    points=sample["points"].to(device).float()
    with torch.no_grad():
        logits,offsets,features=corner_model(points[None])
        corners,scores=predict_corners(points,logits[0],offsets[0],sample["scale"],0.5,corner_radius_m,0.25,2,64)
        if len(corners)<2:
            return corners,scores,np.zeros((0,2),dtype=int),np.zeros((0,),dtype=np.float32)
        node_features=interpolate_features(points,features[0],corners)
        corner_tensor=torch.as_tensor(corners,dtype=torch.float32,device=device)
        score_tensor=torch.as_tensor(scores,dtype=torch.float32,device=device)
        pairs,edge_logits=edge_model(corner_tensor,score_tensor,node_features)
        keep=torch.sigmoid(edge_logits)>=edge_threshold
    return corners,scores,pairs[keep].cpu().numpy(),torch.sigmoid(edge_logits[keep]).cpu().numpy()

In [18]:
test_rows=[]
per_building=[]
for sample in test_set:
    pred_corners,pred_scores,pred_edges,pred_edge_scores=reconstruct(sample,0.5)
    scale=float(sample["scale"])
    truth_corners=sample["corners"][:sample["corner_count"]].numpy()*scale
    truth_edges=sample["edges"][:sample["edge_count"]].numpy()
    counts=reconstruction_counts(pred_corners*scale,pred_edges,truth_corners,truth_edges,0.5)
    test_rows.append(counts)
    per_building.append({"building_id":sample["building_id"],**counts})
test_metrics=finalize_counts(test_rows)
pd.DataFrame([test_metrics])

,ACO,CP,CR,C_F1,EP,ER,E_F1
0,0.190902,0.632379,0.24261,0.350682,0.150399,0.081568,0.105771


In [19]:
metric_rows=[]
for tolerance in [0.1,0.25,0.5]:
    rows=[]
    for sample in test_set:
        pred_corners,pred_scores,pred_edges,pred_edge_scores=reconstruct(sample,0.5)
        scale=float(sample["scale"])
        truth_corners=sample["corners"][:sample["corner_count"]].numpy()*scale
        truth_edges=sample["edges"][:sample["edge_count"]].numpy()
        rows.append(reconstruction_counts(pred_corners*scale,pred_edges,truth_corners,truth_edges,tolerance))
    metric_rows.append({"tolerance_m":tolerance,**finalize_counts(rows)})
metric_table=pd.DataFrame(metric_rows)
metric_table.to_csv(out_root/"test_metrics.csv",index=False)
pd.DataFrame(per_building).to_csv(out_root/"test_per_building.csv",index=False)
metric_table

,tolerance_m,ACO,CP,CR,C_F1,EP,ER,E_F1
0,0.10,0.070017,0.146271,0.056194,0.081195,0.011926,0.006455,0.008377
1,0.25,0.135041,0.460630,0.177764,0.256530,0.088299,0.048247,0.062399
2,0.50,0.190717,0.629274,0.241953,0.349518,0.147092,0.080029,0.103660


In [20]:
export_count=min(100,len(test_set))
export_rows=[]
for i in range(export_count):
    sample=test_set[i]
    corners,scores,edges,edge_scores=reconstruct(sample,0.5)
    center=sample["center"].numpy()
    scale=float(sample["scale"])
    world=corners*scale+center
    base_z=float(world[:,2].min())
    mesh_vertices,mesh_faces=wireframe_to_mesh(world,edges,base_z)
    folder=mesh_dir/sample["building_id"]
    write_obj(folder/"wireframe.obj",world,edges=edges)
    write_obj(folder/"building_mesh.obj",mesh_vertices,faces=mesh_faces)
    np.savez_compressed(folder/"prediction.npz",corners=world,corner_scores=scores,edges=edges,edge_scores=edge_scores)
    export_rows.append({"building_id":sample["building_id"],"corners":len(world),"edges":len(edges),"faces":len(mesh_faces)})
export_frame=pd.DataFrame(export_rows)
export_frame.to_csv(out_root/"mesh_export_index.csv",index=False)
export_frame.head(10)

,building_id,corners,edges,faces
0,5,4,6,10
1,6,4,5,12
2,20,4,6,10
3,31,4,6,10
4,70,4,4,8
5,75,4,4,12
6,76,12,28,59
7,131,4,3,0
8,159,4,6,10
9,192,5,6,16


In [21]:
assert (out_root/"test_metrics.csv").exists()
assert (out_root/"test_per_building.csv").exists()
assert (out_root/"mesh_export_index.csv").exists()
assert all((mesh_dir/x/"wireframe.obj").exists() for x in export_frame.building_id)
assert all((mesh_dir/x/"building_mesh.obj").exists() for x in export_frame.building_id)
run_summary={"corner_epoch":best_corner["epoch"],"edge_epoch":best_edge["epoch"],"test_metrics":metric_rows,"exports":len(export_frame),"corner_threshold":0.5,"edge_threshold":0.5,"corner_radius_m":corner_radius_m,"cluster_radius_m":0.25}
(out_root/"run_summary.json").write_text(json.dumps(run_summary,indent=2))
run_summary

{'corner_epoch': 4,
 'edge_epoch': 4,
 'test_metrics': [{'tolerance_m': 0.1,
   'ACO': 0.07001722890145655,
   'CP': 0.1462714577468676,
   'CR': 0.056194070995377245,
   'C_F1': 0.08119494446572195,
   'EP': 0.01192576618963602,
   'ER': 0.006455378302854515,
   'E_F1': 0.008376554826272547},
  {'tolerance_m': 0.25,
   'ACO': 0.13504080007504438,
   'CP': 0.46062970492884225,
   'CR': 0.17776411213367826,
   'C_F1': 0.25652952246890154,
   'EP': 0.0882990775206344,
   'ER': 0.04824732057585512,
   'E_F1': 0.06239921314775209},
  {'tolerance_m': 0.5,
   'ACO': 0.1907168659882736,
   'CP': 0.629274211339069,
   'CR': 0.2419525849272658,
   'C_F1': 0.3495175371419819,
   'EP': 0.14709228618795306,
   'ER': 0.08002900498744296,
   'E_F1': 0.10365958421625336}],
 'exports': 100,
 'corner_threshold': 0.5,
 'edge_threshold': 0.5,
 'corner_radius_m': 0.35,
 'cluster_radius_m': 0.25}